<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/08_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 — Conviction Dashboard

The dashboard turns the signals and the backtest evidence into a monitoring view: today's
top composite candidates with their per-signal breakdown, the trending names in each
dataset, the member-skill leaderboard, and the validated backtest numbers with their
caveats attached. It reads as a calibrated watchlist that surfaces where to look, with the
honest measurement kept in view.

The interface is a Streamlit app under `dashboard/`. It runs on the bundled mock data with
no key, so anyone can launch it, and a Quiver token switches it to live. This notebook
writes the app into the repo, previews the panels inline, and documents how to deploy it.


## Setup

In [1]:
!pip install -q streamlit quiverquant pandas numpy plotly scipy yfinance

import subprocess, os, sys, logging
from google.colab import userdata

GITHUB_USER = "balla-a12"
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True for live Quiver data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 58.6 MB/s eta 0:00:00
Working in: /content/quant-equity-research


## Write the dashboard app

Three files under `dashboard/`: `panels.py` holds the data layer (kept free of Streamlit so
it stays testable), `app.py` is the interface, and `requirements.txt` pins the deploy
dependencies.


In [2]:
os.makedirs("dashboard", exist_ok=True)
open("dashboard/panels.py", "w").write('"""Data layer for the dashboard, kept free of Streamlit so it can be tested directly.\n\nEvery function returns plain pandas objects or dicts; the Streamlit app in ``app.py``\nrenders them. The adopted configuration from notebook 07 drives the numbers: a\nscore-ranked universe, the dynamic member-skill weighting for the member leaderboard, and\nthe beta/alpha framing for the backtest-evidence panel.\n"""\nimport numpy as np\nimport pandas as pd\n\nfrom quant_research.ingestion.client import QuiverClient\nfrom quant_research.signals.congress import CongressSignal\nfrom quant_research.signals.govcontracts import GovContractsSignal\nfrom quant_research.signals.lobbying import LobbyingSignal\nfrom quant_research.signals.offexchange import OffExchangeSignal\nfrom quant_research.signals.composite import CompositeSignal\nfrom quant_research.signals.member_skill import precompute_buy_returns, member_skill\n\nPARTS = {"congress": 0.40, "gov_contracts": 0.30, "lobbying": 0.20, "off_exchange": 0.10}\n\n\ndef build_client(use_live, token=None, mock_history_days=600):\n    return (QuiverClient(token=token, mock_history_days=mock_history_days) if use_live\n            else QuiverClient(mock=True, mock_history_days=mock_history_days))\n\n\ndef load_signals(client):\n    return {"congress": CongressSignal(client),\n            "gov_contracts": GovContractsSignal(client),\n            "lobbying": LobbyingSignal(client),\n            "off_exchange": OffExchangeSignal(client)}\n\n\ndef composite_ranking(client):\n    """Today\'s blended conviction ranking across all four signals."""\n    signals = load_signals(client)\n    live = CompositeSignal(signals).compute()\n    return live\n\n\ndef trending_by_signal(client, n=8):\n    """Top names in each dataset on its own scale, for the monitoring view."""\n    signals = load_signals(client)\n    out = {}\n    for name, sig in signals.items():\n        r = sig.compute()\n        if len(r):\n            out[name] = r[["score"]].head(n).round(1)\n    return out\n\n\ndef _synthetic_prices(universe, ranks, start="2022-01-01", seed=7):\n    days = pd.bdate_range(pd.Timestamp(start), pd.Timestamp.today())\n    rng = np.random.default_rng(seed)\n    return pd.DataFrame(\n        {t: 100 * np.exp(np.cumsum(rng.normal(\n            sum(0.0012 * (ranks.get(nm, pd.Series()).get(t, 0.5) - 0.5) for nm in ranks),\n            0.010, len(days)))) for t in universe}, index=days)\n\n\ndef member_leaderboard(client, use_live=False, price_n=250, horizon=21, top=8):\n    """Walk-forward member-skill multipliers as of today, top and bottom members.\n\n    Needs a price panel to score matured buys. Live mode pulls real prices; mock mode\n    generates a synthetic panel tied to the signals so the estimate is well defined.\n    """\n    signals = load_signals(client)\n    trades = client.congress_trades(historical=True)\n    ranked = CompositeSignal({k: signals[k] for k in ["congress", "gov_contracts", "lobbying"]}\n                             ).compute()\n    universe = ranked.head(price_n).index.tolist()\n\n    if use_live:\n        from quant_research.backtest.prices import price_history\n        prices = price_history(universe, pd.Timestamp("2022-01-01"),\n                               pd.Timestamp.today().normalize())\n    else:\n        KW = {"congress": "trades", "gov_contracts": "contracts", "lobbying": "filings"}\n        pf = {"congress": trades, "gov_contracts": client.gov_contracts(),\n              "lobbying": client.lobbying()}\n        ranks = {nm: signals[nm].compute(**{KW[nm]: pf[nm]})["score"].rank(pct=True)\n                 for nm in KW}\n        prices = _synthetic_prices(universe, ranks)\n\n    br = precompute_buy_returns(trades, prices, horizon=horizon)\n    mult = member_skill(br, pd.Timestamp.today(), prior_strength=20.0)\n    if not len(mult):\n        return None, 0, 0\n    mult = mult.sort_values(ascending=False)\n    board = pd.concat([mult.head(top), mult.tail(top)]).round(2).to_frame("skill multiplier")\n    return board, len(mult), len(br)\n\n\ndef backtest_evidence():\n    """Validated figures from notebook 07, shown as documented results with caveats."""\n    variants = pd.DataFrame([\n        {"congress variant": "base", "IC (21d)": 0.014, "positive %": 57, "long-short %/mo": 0.260},\n        {"congress variant": "high-conviction", "IC (21d)": 0.014, "positive %": 57, "long-short %/mo": 0.238},\n        {"congress variant": "dynamic member-skill", "IC (21d)": 0.017, "positive %": 59, "long-short %/mo": 0.369},\n    ]).set_index("congress variant")\n    decomposition = {"long_only_cagr": 0.108, "beta": 0.91, "alpha_annual": -0.002, "r_squared": 0.80}\n    caveats = [\n        "Signals are keyed on the disclosure date, so the edge is measured on information "\n        "already public — no look-ahead, and much of the short-horizon timing decays by then.",\n        "The long-only CAGR is roughly 0.91-beta market exposure; the selection alpha is near zero.",\n        "Event-study periods overlap, so the effective independent sample is far below the raw count.",\n        "This is a monitoring and idea-generation view, calibrated to the evidence, rather than a buy list.",\n    ]\n    return variants, decomposition, caveats\n')
open("dashboard/app.py", "w").write('"""Alternative-data conviction dashboard.\n\nA monitoring and idea-generation view over the congressional, government-contract,\nlobbying, and off-exchange signals, with the backtest evidence attached so every number\ncarries its own calibration. Runs on mock data with no key; set a Quiver token to go live.\n\n    streamlit run dashboard/app.py\n"""\nimport os, sys\n_HERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, _HERE)                              # dashboard/ for panels\nsys.path.insert(0, os.path.join(_HERE, "..", "src"))   # repo src for quant_research\n\nimport pandas as pd\nimport plotly.graph_objects as go\nimport streamlit as st\n\nimport panels\n\nst.set_page_config(page_title="Alt-Data Conviction", layout="wide")\n\n\ndef _token():\n    try:\n        return st.secrets.get("QUIVER_API_KEY", os.environ.get("QUIVER_API_KEY"))\n    except Exception:\n        return os.environ.get("QUIVER_API_KEY")\n\n\n@st.cache_data(show_spinner="Loading signals...")\ndef get_ranking(use_live):\n    return panels.composite_ranking(panels.build_client(use_live, _token()))\n\n\n@st.cache_data(show_spinner="Scanning datasets...")\ndef get_trending(use_live):\n    return panels.trending_by_signal(panels.build_client(use_live, _token()))\n\n\n@st.cache_data(show_spinner="Scoring member skill...")\ndef get_board(use_live):\n    return panels.member_leaderboard(panels.build_client(use_live, _token()), use_live=use_live)\n\n\n# ---- sidebar ----------------------------------------------------------------\nst.sidebar.title("Alt-Data Conviction")\nlive = st.sidebar.toggle("Use live Quiver data", value=False,\n                         help="Off uses bundled mock data. On requires a Quiver token in "\n                              "Streamlit secrets or the QUIVER_API_KEY environment variable.")\nif live and not _token():\n    st.sidebar.warning("No token found; staying on mock data.")\n    live = False\nst.sidebar.caption("Mode: " + ("live" if live else "mock"))\nst.sidebar.markdown("---")\nst.sidebar.caption("Signals: congressional trades, government contracts, lobbying, "\n                   "off-exchange volume. Blended into a 0-100 conviction score.")\n\n# ---- header -----------------------------------------------------------------\nst.title("Alternative-Data Conviction Dashboard")\nst.markdown(\n    "A research and idea-generation view over four alternative-data signals. Scores rank "\n    "names by corroborated activity, and the backtest evidence below states how much weight "\n    "each ranking has earned. Read it as a calibrated watchlist that surfaces where to look, "\n    "with the honest measurement kept in view.")\n\nranking = get_ranking(live)\ntrending = get_trending(live)\nboard, n_members, n_buys = get_board(live)\nvariants, decomp, caveats = panels.backtest_evidence()\n\n# ---- composite ranking ------------------------------------------------------\nst.header("Composite conviction ranking")\nleft, right = st.columns([3, 2])\nwith left:\n    cols = ["congress", "gov_contracts", "lobbying", "off_exchange", "n_signals", "score"]\n    st.dataframe(ranking[cols].head(15).round(1), width=\'stretch\')\nwith right:\n    top = ranking.head(12).iloc[::-1]\n    fig = go.Figure()\n    for name, w in panels.PARTS.items():\n        fig.add_bar(y=top.index, x=top[name].fillna(0) * w, name=name, orientation="h")\n    fig.update_layout(barmode="stack", height=420, margin=dict(l=0, r=0, t=10, b=0),\n                      xaxis_title="weighted contribution", legend=dict(orientation="h", y=-0.15))\n    st.plotly_chart(fig, width=\'stretch\')\nst.caption("A tall bar built from several colors is a name corroborated across datasets. "\n           "Absent signals contribute zero, so the score rewards breadth.")\n\n# ---- trending ---------------------------------------------------------------\nst.header("Trending by dataset")\ncc = st.columns(len(trending))\nfor col, (name, tbl) in zip(cc, trending.items()):\n    col.subheader(name.replace("_", " "))\n    col.dataframe(tbl, width=\'stretch\')\n\n# ---- member skill -----------------------------------------------------------\nst.header("Member-skill leaderboard")\nif board is not None:\n    st.caption(f"Walk-forward empirical-Bayes estimate over {n_members} members and "\n               f"{n_buys:,} matured buys. A multiplier above one up-weights that member\'s "\n               "trades; the estimate uses only buys whose window closed as of today.")\n    st.dataframe(board, width=\'stretch\')\nelse:\n    st.info("Not enough matured trades to score members in this mode.")\n\n# ---- backtest evidence ------------------------------------------------------\nst.header("Backtest evidence")\nst.markdown("Validated on the score-ranked universe over 2022 onward, weekly rebalances, "\n            "walk-forward. The dynamic member-skill variant is the adopted configuration.")\nst.dataframe(variants, width=\'stretch\')\n\nm1, m2, m3 = st.columns(3)\nm1.metric("Long-only CAGR", f"{decomp[\'long_only_cagr\']*100:+.1f}%")\nm2.metric("Market beta", f"{decomp[\'beta\']:.2f}", help="Fraction of the return that is market exposure")\nm3.metric("Annualized alpha", f"{decomp[\'alpha_annual\']*100:+.1f}%",\n          help=f"Residual after beta; R^2 {decomp[\'r_squared\']:.2f}")\nst.markdown("The long-only headline compounds mainly through market beta near one; the "\n            "selection alpha after removing it is close to zero. That is the honest reading "\n            "of the double-digit CAGR that vendor trackers advertise.")\n\nwith st.expander("Caveats worth keeping in view"):\n    for c in caveats:\n        st.markdown("- " + c)\n\nst.markdown("---")\nst.caption("Built on the quant-equity-research package. Signals are keyed on disclosure "\n           "dates to stay free of look-ahead. Not investment advice.")\n')
open("dashboard/requirements.txt", "w").write('streamlit>=1.30\npandas\nnumpy\nplotly\nscipy\nyfinance\nquiverquant\n')
print("wrote dashboard/panels.py, dashboard/app.py, dashboard/requirements.txt")

wrote dashboard/panels.py, dashboard/app.py, dashboard/requirements.txt


In [3]:
!pip install -q -e .
import importlib
for p in [os.path.abspath("src"), os.path.abspath("dashboard")]:
    if p not in sys.path:
        sys.path.insert(0, p)
import panels; importlib.reload(panels)
client = panels.build_client(use_live=USE_LIVE, token=QUIVER_TOKEN)
print("panels ready | mode:", "live" if USE_LIVE else "mock")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
panels ready | mode: live


## Preview the panels inline

The same functions the app renders, shown here so the notebook is self-documenting.


### Composite conviction ranking

In [4]:
ranking = panels.composite_ranking(client)
cols = ["congress", "gov_contracts", "lobbying", "off_exchange", "n_signals", "score"]
ranking[cols].head(15).round(1)

https://api.quiverquant.com/beta/live/congresstrading


,congress,gov_contracts,lobbying,off_exchange,n_signals,score
ticker,,,,,,
MSFT,100.0,10.9,53.8,39.7,4,58.0
DELL,NaN,100.0,22.8,54.9,3,40.1
F,NaN,78.9,23.8,56.3,3,34.1
PLTR,0.0,70.5,41.8,42.0,4,33.7
GD,NaN,71.4,55.3,NaN,2,32.5
GM,NaN,17.9,100.0,51.7,3,30.5
MDLN,NaN,68.0,9.3,68.4,3,29.1
T,33.3,7.8,46.1,38.2,4,28.7
INTC,51.8,NaN,12.8,50.0,3,28.3


In [5]:
import plotly.graph_objects as go
top = ranking.head(12).iloc[::-1]
fig = go.Figure()
for name, w in panels.PARTS.items():
    fig.add_bar(y=top.index, x=top[name].fillna(0) * w, name=name, orientation="h")
fig.update_layout(barmode="stack", height=420, title="Conviction stack — top 12",
                  xaxis_title="weighted contribution to score")
fig.show()

### Trending by dataset

In [6]:
trend = panels.trending_by_signal(client, n=6)
for name, tbl in trend.items():
    print(f"--- {name} ---")
    print(tbl.to_string())

https://api.quiverquant.com/beta/live/congresstrading
--- congress ---
        score
ticker       
MSFT    100.0
INTC     51.8
AMD      34.7
T        33.3
FN       32.2
BRK.B    30.0
--- gov_contracts ---
        score
ticker       
DELL    100.0
F        78.9
GD       71.4
PLTR     70.5
MDLN     68.0
TPC      58.0
--- lobbying ---
        score
ticker       
GM      100.0
META     90.8
AMZN     59.7
GOOGL    58.9
MO       58.3
PFE      57.2
--- off_exchange ---
        score
ticker       
ADTX    100.0
BNL      95.3
ABR      94.8
ABTC     94.7
CWK      93.0
AMCR     91.6


### Member-skill leaderboard

The walk-forward empirical-Bayes estimate, top and bottom members as of today.


In [7]:
board, n_members, n_buys = panels.member_leaderboard(client, use_live=USE_LIVE)
print(f"{n_members} members scored from {n_buys:,} matured buys")
board

https://api.quiverquant.com/beta/bulk/congresstrading
https://api.quiverquant.com/beta/live/congresstrading
242 members scored from 18,069 matured buys


,skill multiplier
representative,
A. Mitchell Jr. McConnell,2.72
Bill Flores,2.72
Kathy Castor,2.72
John Fetterman,2.72
John Rutherford,2.72
Nicholas Van Taylor,2.72
Tim Moore,2.72
Randy Neugebauer,2.68
James R. Langevin,0.37


### Backtest evidence

In [8]:
variants, decomp, caveats = panels.backtest_evidence()
print("Congress variants (validated, walk-forward):")
print(variants.to_string())
print(f"\nLong-only CAGR {decomp['long_only_cagr']*100:+.1f}% = beta {decomp['beta']:.2f} "
      f"(R^2 {decomp['r_squared']:.2f}) with annualized alpha {decomp['alpha_annual']*100:+.1f}%")
print("\nCaveats:")
for c in caveats:
    print(" -", c)

Congress variants (validated, walk-forward):
                      IC (21d)  positive %  long-short %/mo
congress variant                                           
base                     0.014          57            0.260
high-conviction          0.014          57            0.238
dynamic member-skill     0.017          59            0.369

Long-only CAGR +10.8% = beta 0.91 (R^2 0.80) with annualized alpha -0.2%

Caveats:
 - Signals are keyed on the disclosure date, so the edge is measured on information already public — no look-ahead, and much of the short-horizon timing decays by then.
 - The long-only CAGR is roughly 0.91-beta market exposure; the selection alpha is near zero.
 - Event-study periods overlap, so the effective independent sample is far below the raw count.
 - This is a monitoring and idea-generation view, calibrated to the evidence, rather than a buy list.


## Run and deploy

Locally:

```
pip install -r dashboard/requirements.txt
streamlit run dashboard/app.py
```

On Streamlit Community Cloud, point a new app at this repository, set the main file to
`dashboard/app.py`, and it runs on mock data immediately. To serve live data, add
`QUIVER_API_KEY` under the app's Secrets. The data layer retries transient rate limits and
falls back gracefully, so the deployed app stays responsive.


## Commit

In [9]:
!git config --global user.email "aballa1234@gmail.com"
!git config --global user.name "Alex Balla"

!git add -A
!git commit -m "Add Streamlit conviction dashboard"
!git push

[main e142b39] Add Streamlit conviction dashboard
 3 files changed, 242 insertions(+)
 create mode 100644 dashboard/app.py
 create mode 100644 dashboard/panels.py
 create mode 100644 dashboard/requirements.txt
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 4.77 KiB | 4.77 MiB/s, done.
Total 6 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/balla-a12/quant-equity-research.git
   ea057d1..e142b39  main -> main


## Recap

The repository now carries the full arc: four alternative-data signals, an honest backtest
that separates beta from alpha, a walk-forward member-skill refinement, and a dashboard that
presents it all with the evidence attached. The dashboard runs on mock for any reviewer and
switches to live with a token, ready to deploy and to link from a portfolio.
